# Notebook 05 — Report Figures & Final Synthesis

## Projet : C-MARL pour la réduction des absences aux rendez-vous médicaux

Ce notebook prépare les éléments finaux à utiliser dans le rapport ou la présentation :

- figures finales ;
- tableau comparatif Easy / Medium / Hard ;
- calcul clair du gain Hard ;
- résumé méthodologique ;
- sauvegarde des outputs dans le dossier `results/`.

Il doit être exécuté après :

1. `01_preprocessing.ipynb`
2. `02_environment.ipynb`
3. `03_agents_training.ipynb`
4. `04_evaluation_results_corrected.ipynb`


## Cellule 1 — Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

## Cellule 2 — Chemins du projet

Si ce notebook est placé dans le dossier `notebooks`, le chemin du projet est automatiquement le dossier parent.

In [ ]:
PROJECT_PATH = os.path.abspath("..")
RESULTS_DIR = os.path.join(PROJECT_PATH, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "final_figures")

os.makedirs(FIGURES_DIR, exist_ok=True)

print("Project path:", PROJECT_PATH)
print("Results dir:", RESULTS_DIR)
print("Figures dir:", FIGURES_DIR)

## Cellule 3 — Charger les résultats d'évaluation

Le fichier `evaluation_comparison.csv` est normalement créé par le Notebook 04.

In [ ]:
evaluation_path = os.path.join(RESULTS_DIR, "evaluation_comparison.csv")

if not os.path.exists(evaluation_path):
    raise FileNotFoundError(
        "Le fichier evaluation_comparison.csv est introuvable. "
        "Exécute d'abord le Notebook 04 corrigé."
    )

evaluation_df = pd.read_csv(evaluation_path)

print("Résultats chargés avec succès.")
evaluation_df

## Cellule 4 — Standardiser les noms de colonnes si nécessaire

Cette cellule rend le notebook plus robuste si les colonnes ont des noms légèrement différents selon la version du Notebook 04.

In [ ]:
print("Colonnes disponibles :")
print(evaluation_df.columns.tolist())

# Normalisation légère des noms de colonnes
evaluation_df.columns = [col.strip() for col in evaluation_df.columns]

# Si la colonne stage est en minuscule ou différemment nommée
possible_stage_cols = ["Stage", "stage"]
for col in possible_stage_cols:
    if col in evaluation_df.columns:
        evaluation_df = evaluation_df.rename(columns={col: "Stage"})
        break

evaluation_df

## Cellule 5 — Identifier les indicateurs disponibles

On détecte automatiquement les colonnes principales pour éviter les erreurs.

In [ ]:
def find_column(possible_names):
    for name in possible_names:
        if name in evaluation_df.columns:
            return name
    return None

avg_reward_col = find_column([
    "Average Reward", "average_reward", "Average_Reward", "avg_reward", "Reward moyen"
])

total_reward_col = find_column([
    "Total Reward", "total_reward", "Total_Reward", "reward_total", "Reward total"
])

good_decision_col = find_column([
    "Good Decision Rate", "good_decision_rate", "Good_Decision_Rate", "Taux bonnes décisions"
])

unnecessary_col = find_column([
    "Unnecessary Intervention Rate", "unnecessary_intervention_rate",
    "Unnecessary_Intervention_Rate", "Taux interventions inutiles"
])

print("Average reward column:", avg_reward_col)
print("Total reward column:", total_reward_col)
print("Good decision column:", good_decision_col)
print("Unnecessary intervention column:", unnecessary_col)

## Cellule 6 — Graphique final : Reward moyenne par stage

Ce graphique montre si le stage Hard obtient le meilleur gain moyen.

In [ ]:
if avg_reward_col is None:
    print("Colonne de reward moyenne introuvable. Graphique ignoré.")
else:
    plt.figure(figsize=(8, 5))
    plt.bar(evaluation_df["Stage"], evaluation_df[avg_reward_col])
    plt.title("Average Reward by Stage")
    plt.xlabel("Stage")
    plt.ylabel("Average Reward")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(FIGURES_DIR, "final_average_reward_by_stage.png")
    plt.savefig(output_path, dpi=300)
    plt.show()

    print("Figure sauvegardée :", output_path)

## Cellule 7 — Graphique final : Reward totale par stage

In [ ]:
if total_reward_col is None:
    print("Colonne de reward totale introuvable. Graphique ignoré.")
else:
    plt.figure(figsize=(8, 5))
    plt.bar(evaluation_df["Stage"], evaluation_df[total_reward_col])
    plt.title("Total Reward by Stage")
    plt.xlabel("Stage")
    plt.ylabel("Total Reward")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(FIGURES_DIR, "final_total_reward_by_stage.png")
    plt.savefig(output_path, dpi=300)
    plt.show()

    print("Figure sauvegardée :", output_path)

## Cellule 8 — Graphique final : Taux de bonnes décisions

Ce graphique est utile dans le rapport pour montrer la performance opérationnelle du système.

In [ ]:
if good_decision_col is None:
    print("Colonne du taux de bonnes décisions introuvable. Graphique ignoré.")
else:
    plt.figure(figsize=(8, 5))
    plt.bar(evaluation_df["Stage"], evaluation_df[good_decision_col])
    plt.title("Good Decision Rate by Stage")
    plt.xlabel("Stage")
    plt.ylabel("Good Decision Rate")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(FIGURES_DIR, "final_good_decision_rate_by_stage.png")
    plt.savefig(output_path, dpi=300)
    plt.show()

    print("Figure sauvegardée :", output_path)

## Cellule 9 — Graphique final : Interventions inutiles

L'objectif est de montrer que le système évite les décisions inutiles, comme envoyer des rappels à des patients qui viennent déjà.

In [ ]:
if unnecessary_col is None:
    print("Colonne du taux d'interventions inutiles introuvable. Graphique ignoré.")
else:
    plt.figure(figsize=(8, 5))
    plt.bar(evaluation_df["Stage"], evaluation_df[unnecessary_col])
    plt.title("Unnecessary Intervention Rate by Stage")
    plt.xlabel("Stage")
    plt.ylabel("Unnecessary Intervention Rate")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(FIGURES_DIR, "final_unnecessary_intervention_rate_by_stage.png")
    plt.savefig(output_path, dpi=300)
    plt.show()

    print("Figure sauvegardée :", output_path)

## Cellule 10 — Calcul du gain Hard

Le sujet demande explicitement `gain Hard`. On calcule donc le gain du stage Hard par rapport aux stages Easy et Medium.

In [ ]:
metric_col = avg_reward_col if avg_reward_col is not None else total_reward_col

if metric_col is None:
    print("Aucune colonne de reward disponible pour calculer le gain Hard.")
else:
    stage_rewards = dict(zip(evaluation_df["Stage"].astype(str).str.lower(), evaluation_df[metric_col]))

    easy_reward = stage_rewards.get("easy")
    medium_reward = stage_rewards.get("medium")
    hard_reward = stage_rewards.get("hard")

    gain_hard_vs_easy = None if easy_reward is None else hard_reward - easy_reward
    gain_hard_vs_medium = None if medium_reward is None else hard_reward - medium_reward

    gain_summary = pd.DataFrame({
        "Comparison": ["Hard - Easy", "Hard - Medium"],
        "Gain": [gain_hard_vs_easy, gain_hard_vs_medium]
    })

    gain_path = os.path.join(RESULTS_DIR, "hard_gain_summary.csv")
    gain_summary.to_csv(gain_path, index=False)

    print("Metric used:", metric_col)
    print("Hard reward:", hard_reward)
    print("Gain Hard vs Easy:", gain_hard_vs_easy)
    print("Gain Hard vs Medium:", gain_hard_vs_medium)
    print("Fichier sauvegardé :", gain_path)

    gain_summary

## Cellule 11 — Charger les rewards d'entraînement

Cette cellule charge les fichiers créés par le Notebook 03 pour visualiser l'apprentissage.

In [ ]:
reward_files = {
    "Easy": os.path.join(RESULTS_DIR, "rewards_easy.csv"),
    "Medium": os.path.join(RESULTS_DIR, "rewards_medium.csv"),
    "Hard": os.path.join(RESULTS_DIR, "rewards_hard.csv")
}

training_rewards = {}

for stage, path in reward_files.items():
    if os.path.exists(path):
        training_rewards[stage] = pd.read_csv(path)
        print(stage, "chargé :", training_rewards[stage].shape)
    else:
        print(stage, "introuvable :", path)

## Cellule 12 — Courbes d'apprentissage des trois stages

Ce graphique montre l'évolution des rewards pendant l'entraînement.

In [ ]:
if not training_rewards:
    print("Aucun fichier de rewards d'entraînement trouvé. Graphique ignoré.")
else:
    plt.figure(figsize=(10, 6))

    for stage, rewards_df in training_rewards.items():
        # Détection de la colonne reward
        reward_col = None
        for col in ["Reward", "reward", "Total Reward", "total_reward", "Episode Reward", "episode_reward"]:
            if col in rewards_df.columns:
                reward_col = col
                break

        if reward_col is None:
            # Si le fichier contient une seule colonne, on l'utilise
            if rewards_df.shape[1] == 1:
                reward_col = rewards_df.columns[0]
            else:
                print("Colonne reward introuvable pour", stage, ":", rewards_df.columns.tolist())
                continue

        plt.plot(rewards_df[reward_col].values, label=stage)

    plt.title("Training Rewards by Stage")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(FIGURES_DIR, "final_training_rewards_by_stage.png")
    plt.savefig(output_path, dpi=300)
    plt.show()

    print("Figure sauvegardée :", output_path)

## Cellule 13 — Générer un résumé textuel pour le rapport

Ce fichier `.txt` peut être utilisé pour rédiger la partie méthodologie et résultats du rapport.

In [ ]:
summary_lines = []
summary_lines.append("Projet : C-MARL pour la réduction des absences aux rendez-vous médicaux")
summary_lines.append("")
summary_lines.append("1. Objectif")
summary_lines.append("L'objectif est de simuler un système multi-agents coopératif capable d'améliorer la gestion des rendez-vous médicaux.")
summary_lines.append("Les agents apprennent à choisir des actions comme ne rien faire, envoyer un rappel, prioriser un patient ou proposer une reprogrammation.")
summary_lines.append("")
summary_lines.append("2. Dataset")
summary_lines.append("Le projet utilise le dataset Medical Appointment No Shows. Chaque ligne représente un rendez-vous médical.")
summary_lines.append("La variable No_show indique si le patient s'est présenté ou non.")
summary_lines.append("")
summary_lines.append("3. Stages")
summary_lines.append("Stage Easy : variables limitées, actions simples.")
summary_lines.append("Stage Medium : ajout de variables médicales.")
summary_lines.append("Stage Hard : ajout de variables plus complètes comme WaitingGroup, AppointmentWeekday et Neighbourhood.")
summary_lines.append("")
summary_lines.append("4. Méthode")
summary_lines.append("L'approche repose sur un Q-learning discret, avec une Q-table sauvegardée sous forme de checkpoint pour chaque stage.")
summary_lines.append("")
summary_lines.append("5. Résultats")
summary_lines.append(evaluation_df.to_string(index=False))

if 'gain_summary' in globals():
    summary_lines.append("")
    summary_lines.append("6. Gain Hard")
    summary_lines.append(gain_summary.to_string(index=False))

summary_lines.append("")
summary_lines.append("Conclusion :")
summary_lines.append("Le stage Hard est censé obtenir le meilleur gain car il utilise davantage d'informations pour guider la décision des agents.")

summary_text = "\n".join(summary_lines)

summary_path = os.path.join(RESULTS_DIR, "final_report_synthesis.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("\nRésumé sauvegardé :", summary_path)


## Cellule 14 — Liste des fichiers générés

Cette cellule vérifie les outputs créés par ce notebook.

In [ ]:
print("Fichiers dans final_figures:")
for file in os.listdir(FIGURES_DIR):
    print("-", file)

print("\nFichiers importants dans results:")
for file in os.listdir(RESULTS_DIR):
    if file.endswith(".csv") or file.endswith(".txt"):
        print("-", file)


## Cellule 15 — Fin du Notebook 05

In [ ]:
print("Notebook 05 terminé avec succès.")
print("Les figures finales et la synthèse sont prêtes pour le rapport et la présentation.")